In [1]:
from openai import OpenAI

In [2]:
client = OpenAI()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [3]:
assistant = client.beta.assistants.create(
    model="gpt-3.5-turbo",
    name="数学助手",
    instructions="你是一个数学助手，可以通过编写和运行代码来回答数学相关问题。",
    tools=[{"type": "code_interpreter"}]
)

In [4]:
thread = client.beta.threads.create()

In [5]:
thread.id

'thread_qjyG7TTbIDpvcNQAdFO9KiMj'

In [6]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="我需要解这个方程`5x^2−1200x+72000=0，未知数应该是多少？"
)

In [7]:
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
    instructions="请称呼用户为粒粒"
)

In [8]:
run

Run(id='run_UqgpuliqyeJG7ro1VCaB2hBS', assistant_id='asst_jtLgtWj5ee3K4PyzGf3i4MJD', cancelled_at=None, completed_at=None, created_at=1713238485, expires_at=1713239085, failed_at=None, file_ids=[], instructions='请称呼用户为粒粒', last_error=None, metadata={}, model='gpt-3.5-turbo', object='thread.run', required_action=None, started_at=None, status='queued', thread_id='thread_qjyG7TTbIDpvcNQAdFO9KiMj', tools=[CodeInterpreterTool(type='code_interpreter')], usage=None, temperature=1.0, top_p=1.0, max_completion_tokens=None, max_prompt_tokens=None, truncation_strategy={'type': 'auto', 'last_messages': None}, incomplete_details=None, response_format='auto', tool_choice='auto')

In [9]:
client.beta.threads.runs.retrieve(
    thread_id=thread.id,
    run_id=run.id
).status

'completed'

In [10]:
while run.status != "completed":
    keep_retrieving_run = client.beta.threads.runs.retrieve(
        thread_id=thread.id,
        run_id=run.id
    )
    print(f"运行状态：{keep_retrieving_run.status}")
    if keep_retrieving_run.status == "completed":
        break

运行状态：completed


In [11]:
messages = client.beta.threads.messages.list(
    thread_id=thread.id
)

In [12]:
messages.data

[Message(id='msg_CATnOIPcyBdnTwUDnzoS6P94', assistant_id='asst_jtLgtWj5ee3K4PyzGf3i4MJD', completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='经过计算，方程5x^2−1200x+72000=0的根是x=120。因此，未知数x的值为120。如果你有任何其他问题或需要进一步帮助，请随时告诉我。'), type='text')], created_at=1713238498, file_ids=[], incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_UqgpuliqyeJG7ro1VCaB2hBS', status=None, thread_id='thread_qjyG7TTbIDpvcNQAdFO9KiMj'),
 Message(id='msg_5u0uAHuR9NDRUjKznfxZoyz7', assistant_id='asst_jtLgtWj5ee3K4PyzGf3i4MJD', completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='你想要解这个二次方程来找出未知数x的值是吧。我可以帮你用求根公式来找到方程的根。让我们开始计算吧。'), type='text')], created_at=1713238486, file_ids=[], incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_UqgpuliqyeJG7ro1VCaB2hBS', status=None, thread_id='thread_qjyG7TTbIDpvcNQAdFO9KiMj'),
 Message(id='msg_i3D0A5qqm3ORY6Nx

In [13]:
for data in messages.data:
    print(data.content[0].text.value)
    print("------")

经过计算，方程5x^2−1200x+72000=0的根是x=120。因此，未知数x的值为120。如果你有任何其他问题或需要进一步帮助，请随时告诉我。
------
你想要解这个二次方程来找出未知数x的值是吧。我可以帮你用求根公式来找到方程的根。让我们开始计算吧。
------
我需要解这个方程`5x^2−1200x+72000=0，未知数应该是多少？
------


In [14]:
def get_response_from_assistant(assistant, thread, prompt, run_instruction=""):
    message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content=prompt
    )
    
    run = client.beta.threads.runs.create(
      thread_id=thread.id,
      assistant_id=assistant.id,
      instructions=run_instruction
    )
    
    while run.status != "completed":
        keep_retrieving_run = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id
        )
        print(f"Run status: {keep_retrieving_run.status}")

        if keep_retrieving_run.status == "completed":
            break
    
    messages = client.beta.threads.messages.list(
        thread_id=thread.id
    )
    
    for data in messages.data:
        print("\n")
        print(data.content[0].text.value)
        print("------")

In [15]:
get_response_from_assistant(assistant, thread, "2的56次方等于多少")

Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: in_progress
Run status: completed


2的56次方等于72057594037927936。如果你有任何其他问题，欢迎随时向我提问。
------


要计算2的56次方，只需要将2乘以自身共56次。让我们计算一下。
------


2的56次方等于多少
------


经过计算，方程5x^2−1200x+72000=0的根是x=120。因此，未知数x的值为120。如果你有任何其他问题或需要进一步帮助，请随时告诉我。
------


你想要解这个二次方程来找出未知数x的值是吧。我可以帮你用求根公式来找到方程的根。让我们开始计算吧。
------


我需要解这个方程`5x^2−1200x+72000=0，未知数应该是多少？
------
